# Phase 3: Optimized GPU with Sigmoid + BCE Loss (Kaggle Version)
**CSC14120 - Parallel Programming**

---

## Cải tiến so với Phase 3 gốc (MSE Loss)

### Thay đổi Architecture:
- **Output Activation:** Thêm Sigmoid sau Conv5 cuối cùng
- **Loss Function:** Binary Cross-Entropy (BCE) thay vì MSE

### Lý do sử dụng Sigmoid + BCE:

| Aspect | MSE (Phase 3 gốc) | BCE + Sigmoid (Version này) |
|:-------|:------------------|:----------------------------|
| Output range | Unbounded (-∞, +∞) | Bounded [0, 1] |
| Gradient behavior | Linear | Stronger near 0.5, weaker near 0/1 |
| Pixel validity | May need clipping | Always valid |
| Theory | Regression | Probabilistic reconstruction |

### BCE Loss Formula:
```
BCE = -[y*log(ŷ + ε) + (1-y)*log(1-ŷ + ε)] / N

where:
  y = target pixel value (normalized to [0,1])
  ŷ = sigmoid(model_output)
  ε = 1e-7 (numerical stability)
  N = total number of pixels
```

### Sigmoid Activation:
```
sigmoid(x) = 1 / (1 + exp(-x))

Numerically stable version:
  if x >= 0: sigmoid(x) = 1 / (1 + exp(-x))
  if x < 0:  sigmoid(x) = exp(x) / (1 + exp(x))
```

---

## Hướng dẫn chạy trên Kaggle:
1. Upload project lên Kaggle Dataset
2. Tạo notebook mới và add dataset
3. Chạy tất cả cells

In [ ]:
# Kiểm tra GPU
!nvidia-smi
!nvcc --version

In [ ]:
# Copy project từ Kaggle dataset
import os
import glob
import shutil

input_dir = '/kaggle/input'
src_dirs = glob.glob(f'{input_dir}/**/src', recursive=True)

if src_dirs:
    project_dir = os.path.dirname(src_dirs[0])
    print(f"Found project at: {project_dir}")
    shutil.copytree(project_dir, '/kaggle/working/project', dirs_exist_ok=True)
    
    for root, dirs, _ in os.walk('/kaggle/working/project'):
        if 'src' in dirs:
            os.chdir(root)
            break
else:
    print("ERROR: Project not found")

print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# Download CIFAR-10
import urllib.request
import tarfile

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve(
        'https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 
        'data/cifar.tar.gz'
    )
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/

print('CIFAR-10 ready!')

## Build Phase 3 với BCE Loss

**Compiler flags:**
- `-O3`: Maximum optimization
- `-DUSE_OPTIMIZED_KERNELS`: Enable cuDNN path
- `--use_fast_math`: Fast math operations
- `-lcublas -lcudnn`: Link cuBLAS and cuDNN

**Runtime flag:**
- `--bce-loss`: Enable BCE loss with Sigmoid output

In [ ]:
# Build Phase 3 with cuDNN (BCE is runtime flag)
# Note: Kaggle uses sm_70 for Tesla P100 or sm_75 for Tesla T4
!nvcc -O3 -std=c++17 -arch=sm_70 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -Iinclude -lcublas -lcudnn \
    -o gpu_train_bce \
    src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp

print('Build complete with cuDNN + BCE Loss support!')

## Training với BCE Loss

**Lưu ý quan trọng:**
- BCE loss values thường **cao hơn** MSE loss (khác scale)
- Không so sánh trực tiếp BCE loss với MSE loss
- Đánh giá qua reconstruction quality và downstream accuracy

In [ ]:
# Train với BCE Loss + Sigmoid
!./gpu_train_bce --data data --epochs 20 --batch 64 --lr 0.001 \
    --bce-loss \
    --log phase3_bce.csv --log-txt phase3_bce.txt --save-weights phase3_bce.weights

## Kết quả và Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('phase3_bce.csv')
ep = df[df['batch'].isna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ep['epoch'], ep['loss'], 'purple', marker='o')
ax1.set_title('BCE Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss')
ax1.grid(True)

ax2.plot(ep['epoch'], ep['epoch_time_sec'], 'green', marker='o')
ax2.set_title('Time per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Time (s)')
ax2.grid(True)

plt.tight_layout()
plt.savefig('phase3_bce_results.png', dpi=150)
plt.show()

print("="*50)
print("PHASE 3 BCE LOSS RESULTS (Kaggle)")
print("="*50)
print(f"Best BCE Loss: {ep['best_loss'].iloc[-1]:.6f}")
print(f"Final BCE Loss: {ep['loss'].iloc[-1]:.6f}")
print(f"Avg Time per Epoch: {ep['epoch_time_sec'].mean():.2f}s")
print(f"Total Training Time: {ep['epoch_time_sec'].sum():.2f}s ({ep['epoch_time_sec'].sum()/60:.2f} min)")
print("="*50)

In [ ]:
# Show training log
print("="*60)
print("TRAINING LOG (last 30 lines)")
print("="*60)
!tail -30 phase3_bce.txt

In [ ]:
# Verify weights file structure
import struct
import numpy as np

path = "phase3_bce.weights"

with open(path, "rb") as f:
    header = f.read(12)
    magic, version, num_layers = struct.unpack("III", header)
    print(f"MAGIC = {hex(magic)}, version = {version}, num_layers = {num_layers}")

    layers = []
    for li in range(num_layers):
        in_c, out_c, k = struct.unpack("iii", f.read(12))
        print(f"\nLayer {li}: in_c={in_c}, out_c={out_c}, k={k}")

        (w_size,) = struct.unpack("i", f.read(4))
        w_bytes = f.read(4 * w_size)
        w = np.frombuffer(w_bytes, dtype=np.float32)
        print(f"  weights: shape=({out_c}, {in_c}, {k}, {k}), w_size={w_size}")
        print("  first 5 weights:", w[:5])

        (b_size,) = struct.unpack("i", f.read(4))
        b_bytes = f.read(4 * b_size)
        b = np.frombuffer(b_bytes, dtype=np.float32)
        print(f"  bias: shape=({b_size},)")
        print("  first 5 biases:", b[:5])

        layers.append(((in_c, out_c, k), w, b))

## So sánh MSE vs BCE Loss

### Khi nào dùng BCE + Sigmoid:
- ✅ Muốn output luôn trong [0, 1] (valid pixel values)
- ✅ Muốn model học "probability" của pixel intensity
- ✅ Training từ đầu (không dùng pretrained weights)

### Khi nào dùng MSE (không Sigmoid):
- ✅ Đã có pretrained weights với MSE
- ✅ Muốn gradient flow mạnh hơn ở output layer
- ✅ Chấp nhận clip output khi visualize

### Lưu ý:
- Weights từ MSE model **KHÔNG tương thích** với BCE model
- Phải train lại từ đầu khi chuyển đổi loss function